In [1]:
from selenium import webdriver
from selenium.webdriver.support.ui import Select
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException
import pandas as pd
import time
import os

# Department mapping: "Dropdown text": "option value"
departments = {
    "Water Resources Department": "150",
    "WATCO Bhubaneswar": "149",
    "Rural Water Supply and Sanitation": "137"
}

all_data = []

driver = webdriver.Chrome()  # Make sure chromedriver is in PATH

for dept_name, dept_value in departments.items():
    print(f"\nScraping for department: {dept_name}")
    url = "https://tendersodisha.gov.in/nicgep/app?page=WebTenderStatusLists&service=page"
    driver.get(url)
    time.sleep(2)
    
    # Select the department from dropdown
    select = Select(driver.find_element(By.ID, "OrganName"))
    select.select_by_value(dept_value)
    time.sleep(1)
    
    # Handle captcha - show to user and input manually
    captcha_img = driver.find_element(By.ID, "captchaImage").screenshot_as_png
    with open('captcha.png', 'wb') as f:
        f.write(captcha_img)
    try:
        os.startfile('captcha.png')  # Opens image for user (Windows)
    except:
        print("Open 'captcha.png' file to see CAPTCHA and type the value.")
    captcha_value = input("Enter CAPTCHA for {}: ".format(dept_name))
    captcha_input = driver.find_element(By.ID, "captchaText")
    captcha_input.send_keys(captcha_value)
    driver.find_element(By.ID, "Search").click()
    time.sleep(3)
    
    # Scrape all result pages
    while True:
        # Scrape table rows except header
        rows = driver.find_elements(By.CSS_SELECTOR, "#tabList tr")
        for row in rows[1:]:  # Skip header row
            tds = row.find_elements(By.TAG_NAME, "td")
            if len(tds) >= 6:
                all_data.append({
                    "Department": dept_name,
                    "S.No": tds[0].text.strip(),
                    "Tender ID": tds[1].text.strip(),
                    "Title and Ref.No.": tds[2].text.strip(),
                    "Organisation Chain": tds[3].text.strip(),
                    "Tender Stage": tds[4].text.strip(),
                    "Status": tds[5].text.strip()
                })
        # Try to click "Next >"
        try:
            next_btn = driver.find_element(By.LINK_TEXT, 'Next >')
            next_btn.click()
            time.sleep(2)
        except NoSuchElementException:
            break  # No more pages

driver.quit()

# Save to Excel
df = pd.DataFrame(all_data)
df.to_excel("odisha_tenders_three_departments.xlsx", index=False)
print("All tenders saved to odisha_tenders_three_departments.xlsx")



Scraping for department: Water Resources Department


Enter CAPTCHA for Water Resources Department:  DCB7zH



Scraping for department: WATCO Bhubaneswar


Enter CAPTCHA for WATCO Bhubaneswar:  BhTss7



Scraping for department: Rural Water Supply and Sanitation


Enter CAPTCHA for Rural Water Supply and Sanitation:  8TEX6t


All tenders saved to odisha_tenders_three_departments.xlsx
